## Import data

In [1]:
import pandas as pd
import os
import opendatasets as od

In [2]:
od.download("https://www.kaggle.com/datasets/nicoletacilibiu/movies-and-ratings-for-recommendation-system", data_dir="data")

Skipping, found downloaded files in "data/movies-and-ratings-for-recommendation-system" (use force=True to force download)


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when

In [4]:
spark = SparkSession.builder.appName("Movies Dataset").getOrCreate()

movies_df = spark.read.csv("data/movies-and-ratings-for-recommendation-system/movies.csv", header=True, inferSchema=True)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/26 15:32:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
ratings_df = spark.read.csv("data/movies-and-ratings-for-recommendation-system/ratings.csv", header=True, inferSchema=True)

## Clean data

In [6]:
movies_df.dtypes

[('movieId', 'int'), ('title', 'string'), ('genres', 'string')]

In [7]:
ratings_df.dtypes

[('userId', 'int'),
 ('movieId', 'int'),
 ('rating', 'double'),
 ('timestamp', 'int')]

In [8]:
ratings_df = ratings_df.drop("timestamp")

## Fitting the Alternating Least Squares Model

In [9]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS 

In [10]:
(training_df, testing_df) = ratings_df.randomSplit([0.8, 0.2], seed=42)

als = ALS(
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop"  # Avoid NaN during evaluation
)

als_model = als.fit(training_df)

24/11/26 15:32:21 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/11/26 15:32:21 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
24/11/26 15:32:21 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [11]:
predictions = als_model.transform(testing_df)
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)

In [12]:
print(f"Root-mean-square error = {rmse}")

Root-mean-square error = 0.8823107922070733


## Cross-validation to Find the Optimal Model

In [13]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Create the parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(als.rank, [4, 10, 50]) \
    .addGrid(als.regParam, [0.01, 0.001, 0.1]) \
    .build()

# Instantiate the cross-validator
crossval = CrossValidator(
    estimator=als,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3  # 3-fold cross-validation
)

# Perform cross-validation to find the best model
cv_model = crossval.fit(training_df)

best_model = cv_model.bestModel

24/11/26 15:32:30 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [14]:
# Display the best rank and regularization parameter
print(f"Best rank: {best_model.rank}")
print(f"Best regularization parameter: {best_model._java_obj.parent().getRegParam()}")

# Evaluate the best model on the testing set
predictions = best_model.transform(testing_df)
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error of the best model = {rmse}")

Best rank: 50
Best regularization parameter: 0.1
Root-mean-square error of the best model = 0.8747986400568157


In [15]:
def name_retriever(movie_id, movie_title_df):
    # Filter the movie_title_df to find the row that matches the movie_id
    movie = movie_title_df.filter(movie_title_df.movieId == movie_id).select("title").collect()
    
    # Return the movie title as a string
    if movie:
        return movie[0]['title']
    else:
        return None


In [16]:
def get_recommendations_for_user(user_id, als_model, ratings_df, movies_df, num_recs=10):
    
    user_df = ratings_df.select("userId").filter(f"userId = {user_id}").distinct()
    
    user_recs = als_model.recommendForUserSubset(user_df, num_recs)

    recommendations = user_recs.select("recommendations").collect()[0]["recommendations"]
    
    recommendation_list = []
    
    for idx, rec in enumerate(recommendations, start=1):
        movie_title = name_retriever(rec["movieId"], movies_df)
        predicted_score = rec["rating"]
        recommendation_list.append({
            "rank": idx,
            "movieId": rec["movieId"],
            "title": movie_title,
            "predicted_score": round(predicted_score, 6)
        })
    
    return recommendation_list

In [17]:
user_id = 1
recommendations = get_recommendations_for_user(user_id, als_model, ratings_df, movies_df, num_recs=10)
for rec in recommendations:
    print(f"Recommendation {rec['rank']}: {rec['title']}  | predicted score :{rec['predicted_score']}")

Recommendation 1: Yojimbo (1961)  | predicted score :5.712624
Recommendation 2: Seve (2014)  | predicted score :5.617299
Recommendation 3: Victory (a.k.a. Escape to Victory) (1981)  | predicted score :5.602072
Recommendation 4: Dragon Ball Z: The History of Trunks (Doragon bôru Z: Zetsubô e no hankô!! Nokosareta chô senshi - Gohan to Torankusu) (1993)  | predicted score :5.539546
Recommendation 5: On the Beach (1959)  | predicted score :5.539546
Recommendation 6: Rules of Attraction, The (2002)  | predicted score :5.525299
Recommendation 7: Saving Face (2004)  | predicted score :5.521188
Recommendation 8: Belle époque (1992)  | predicted score :5.480838
Recommendation 9: On the Town (1949)  | predicted score :5.475943
Recommendation 10: Wallace & Gromit: The Best of Aardman Animation (1996)  | predicted score :5.472951


## Getting Predictions for a New User

In [19]:
from pyspark.sql import Row

In [20]:
def get_recommendations_for_new_user(user_id, new_ratings, rating_df, movie_title_df, num_recs=10):
    # Step 1: Convert the new_ratings list into a Spark DataFrame
    new_ratings_df = spark.createDataFrame(
        [Row(userId=rating[0], movieId=rating[1], rating=rating[2]) for rating in new_ratings]
    )

    # Step 2: Combine the new ratings DataFrame with the original ratings DataFrame
    updated_ratings_df = rating_df.union(new_ratings_df)
    
    # Step 3: Create and fit an ALS model
    als = ALS(
        maxIter=10,
        regParam=0.1,
        rank=50,
        userCol="userId",
        itemCol="movieId",
        ratingCol="rating",
        coldStartStrategy="drop"
    )
    model = als.fit(updated_ratings_df)

    # Step 4: Make recommendations for all users
    user_recommendations = model.recommendForAllUsers(num_recs)
    
    # Step 5: Get recommendations specifically for the new user
    new_user_recommendations = user_recommendations.filter(user_recommendations.userId == user_id).collect()
    
    # Step 6: Extract the top recommendations
    if new_user_recommendations:
        recommendations = new_user_recommendations[0]["recommendations"]
        movie_ids = [rec["movieId"] for rec in recommendations]
        
        # Retrieve movie titles using the name_retriever function
        recommended_movies = [name_retriever(movie_id, movie_title_df) for movie_id in movie_ids]
        
        # Print the recommendations in a reader-friendly format
        print(f"Top {num_recs} movie recommendations for User {user_id}:")
        for idx, movie in enumerate(recommended_movies, start=1):
            print(f"{idx}. {movie}")
    else:
        print(f"No recommendations found for User {user_id}.")

## Content-Based Filtering

In [21]:
from pyspark.sql.functions import col, split, explode
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler
import numpy as np

In [ ]:
# movies_df = movies_df.withColumn("genres", split(col("genres"), "\|"))

In [ ]:
# exploded_movies_df = movies_df.withColumn("genre", explode(col("genres")))
# exploded_movies_df.show(10)

+-------+--------------------+--------------------+---------+
|movieId|               title|              genres|    genre|
+-------+--------------------+--------------------+---------+
|      1|    Toy Story (1995)|[Adventure, Anima...|Adventure|
|      1|    Toy Story (1995)|[Adventure, Anima...|Animation|
|      1|    Toy Story (1995)|[Adventure, Anima...| Children|
|      1|    Toy Story (1995)|[Adventure, Anima...|   Comedy|
|      1|    Toy Story (1995)|[Adventure, Anima...|  Fantasy|
|      2|      Jumanji (1995)|[Adventure, Child...|Adventure|
|      2|      Jumanji (1995)|[Adventure, Child...| Children|
|      2|      Jumanji (1995)|[Adventure, Child...|  Fantasy|
|      3|Grumpier Old Men ...|   [Comedy, Romance]|   Comedy|
|      3|Grumpier Old Men ...|   [Comedy, Romance]|  Romance|
+-------+--------------------+--------------------+---------+
only showing top 10 rows



In [ ]:
# # Use StringIndexer to convert genres to numeric values
# indexer = StringIndexer(inputCol="genre", outputCol="genre_index")
# indexed_movies_df = indexer.fit(exploded_movies_df).transform(exploded_movies_df)

In [ ]:
# # One-hot encoding of genres
# encoder = OneHotEncoder(inputCol="genre_index", outputCol="genre_vector")
# encoded_movies_df = encoder.fit(indexed_movies_df).transform(indexed_movies_df)

In [ ]:
# encoded_movies_df.show(10)

+-------+--------------------+--------------------+---------+-----------+---------------+
|movieId|               title|              genres|    genre|genre_index|   genre_vector|
+-------+--------------------+--------------------+---------+-----------+---------------+
|      1|    Toy Story (1995)|[Adventure, Anima...|Adventure|        5.0| (19,[5],[1.0])|
|      1|    Toy Story (1995)|[Adventure, Anima...|Animation|       11.0|(19,[11],[1.0])|
|      1|    Toy Story (1995)|[Adventure, Anima...| Children|       10.0|(19,[10],[1.0])|
|      1|    Toy Story (1995)|[Adventure, Anima...|   Comedy|        1.0| (19,[1],[1.0])|
|      1|    Toy Story (1995)|[Adventure, Anima...|  Fantasy|        9.0| (19,[9],[1.0])|
|      2|      Jumanji (1995)|[Adventure, Child...|Adventure|        5.0| (19,[5],[1.0])|
|      2|      Jumanji (1995)|[Adventure, Child...| Children|       10.0|(19,[10],[1.0])|
|      2|      Jumanji (1995)|[Adventure, Child...|  Fantasy|        9.0| (19,[9],[1.0])|
|      3|G

In [ ]:
# from pyspark.sql import functions as F
# aggregated_df = encoded_movies_df.groupBy("movieId").agg(F.collect_list("genre_vector").alias("genre_vectors"))

In [ ]:
# aggregated_df.show(10)

+-------+--------------------+
|movieId|       genre_vectors|
+-------+--------------------+
|      1|[(19,[5],[1.0]), ...|
|      2|[(19,[5],[1.0]), ...|
|      3|[(19,[1],[1.0]), ...|
|      4|[(19,[1],[1.0]), ...|
|      5|    [(19,[1],[1.0])]|
|      6|[(19,[3],[1.0]), ...|
|      7|[(19,[1],[1.0]), ...|
|      8|[(19,[5],[1.0]), ...|
|      9|    [(19,[3],[1.0])]|
|     10|[(19,[3],[1.0]), ...|
+-------+--------------------+
only showing top 10 rows



In [ ]:
# def cosine_similarity(vec1, vec2):
#     print(len(vec1), len(vec2))
#     dot_product = float(Vectors.dense(vec1).dot(Vectors.dense(vec2)))
#     norm1 = float(np.linalg.norm(np.array(vec1)))
#     norm2 = float(np.linalg.norm(np.array(vec2)))
#     if norm1 > 0.0 and norm2 > 0.0:
#         return dot_product / (norm1 * norm2)
#     else:
#         return 0.0

In [ ]:
# def content_based_recommendation(movie_id, movie_df, top_n=5):
#     # Get the genre vector of the target movie
#     target_movie = movie_df.filter(col("movieId") == movie_id).collect()[0]
#     genre_vectors = target_movie['genre_vectors']
    
#     # Find the most similar movies based on genre vectors (cosine similarity)
#     similar_movies = movie_df.filter(col("movieId") != movie_id).collect()
#     similarity_scores = []
#     for similar_movie in similar_movies:
#         score = cosine_similarity(genre_vectors, similar_movie['genre_vectors'])
#         similarity_scores.append((similar_movie['movieId'], score))
    
#     # Sort by similarity score and get the top N similar movies
#     similar_movies_sorted = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
#     # Return the top N similar movies
#     return similar_movies_sorted[:top_n]

In [27]:
from pyspark.ml.feature import CountVectorizer
from pyspark.sql import functions as F
def build_genre_features(movies_df):
    """
    This function extracts the genre features and creates a vector representation
    for each movie's genre.
    """
    # Split the genre column into individual genres (assuming genres are separated by '|')
    movies_df = movies_df.withColumn("genres", F.split(movies_df["genres"], "\|"))
    
    # Create a count vectorizer to convert genres to a vector format (one-hot encoding)
    count_vectorizer = CountVectorizer(inputCol="genres", outputCol="genre_vector")
    model = count_vectorizer.fit(movies_df)
    
    # Apply the model to generate genre vectors
    movies_with_genres = model.transform(movies_df)
    
    return movies_with_genres

In [28]:
from sklearn.metrics.pairwise import cosine_similarity
def calculate_genre_similarity(movies_with_genres, movie_id, top_n=10):
    """
    Calculates cosine similarity between the input movie and all other movies based on genres.
    """
    # Get the genre vector of the input movie
    movie_vector = movies_with_genres.filter(f"movieId = {movie_id}").select("genre_vector").collect()[0]["genre_vector"]
    
    # Convert all movie vectors into a list
    all_vectors = movies_with_genres.select("movieId", "genre_vector").collect()
    movie_vectors = [(row["movieId"], row["genre_vector"].toArray()) for row in all_vectors]
    
    # Compute cosine similarity between the input movie and all other movies
    movie_ids, vectors = zip(*movie_vectors)
    cosine_sim = cosine_similarity([movie_vector.toArray()], np.array(vectors))[0]
    
    # Get top N most similar movies
    similar_movies = sorted(zip(movie_ids, cosine_sim), key=lambda x: x[1], reverse=True)[:top_n]
    
    return similar_movies

In [29]:
def get_content_based_recommendations_for_user(user_id, ratings_df, movies_df, num_recs=10):
    """
    Content-based recommendation system using movie genres.
    """
    # Build genre vectors for all movies
    movies_with_genres = build_genre_features(movies_df)
    
    # Get movies rated by the user
    user_ratings = ratings_df.filter(f"userId = {user_id}")
    
    recommended_movies = []
    
    for row in user_ratings.collect():
        movie_id = row["movieId"]
        
        # Get similar movies based on genre
        similar_movies = calculate_genre_similarity(movies_with_genres, movie_id, top_n=num_recs)
        
        # Add these similar movies to the recommendations list
        for sim_movie_id, sim_score in similar_movies:
            if sim_movie_id != movie_id:  # Avoid recommending the same movie
                movie_title = movies_df.filter(f"movieId = {sim_movie_id}").select("title").collect()[0]["title"]
                recommended_movies.append({
                    "userId": user_id,
                    "movieId": sim_movie_id,
                    "title": movie_title,
                    "predicted_score": round(sim_score, 6)
                })
    
    # Sort by predicted score and return top N recommendations
    recommended_movies = sorted(recommended_movies, key=lambda x: x["predicted_score"], reverse=True)[:num_recs]
    
    return recommended_movies

In [30]:
user_id = 1 
ratings_df = ratings_df  
movies_df = movies_df 

# Get content-based recommendations for the user
recommendations = get_content_based_recommendations_for_user(user_id, ratings_df, movies_df, num_recs=10)

# Display the recommendations
for rec in recommendations:
    print(f"Movie: {rec['title']}, Predicted Score: {rec['predicted_score']}")


Movie: Antz (1998), Predicted Score: 1.0
Movie: Toy Story 2 (1999), Predicted Score: 1.0
Movie: Adventures of Rocky and Bullwinkle, The (2000), Predicted Score: 1.0
Movie: Emperor's New Groove, The (2000), Predicted Score: 1.0
Movie: Monsters, Inc. (2001), Predicted Score: 1.0
Movie: Wild, The (2006), Predicted Score: 1.0
Movie: Shrek the Third (2007), Predicted Score: 1.0
Movie: Tale of Despereaux, The (2008), Predicted Score: 1.0
Movie: Asterix and the Vikings (Astérix et les Vikings) (2006), Predicted Score: 1.0
Movie: Sabrina (1995), Predicted Score: 1.0


In [25]:
movies_df.show()

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
|     11|American Presiden...|Comedy|Drama|Romance|
|     12|Dracula: Dead and...|       Comedy|Horror|
|     13|        Balto (1995)|Adventure|Animati...|
|     14|        Nixon (1995)|               Drama|
|     15|Cutthroat Island ...|Action|Adventure|...|
|     16|       Casino (1995)|         Crime|Drama|
|     17|Sen